# Nutrition5k: resultados de los transformers

No entrena nada: lee los runs de `outputs/transformer/` (los escribe `02_experiments.ipynb` de esta carpeta) y los
resume como `notebooks/mobilenet/03_results.ipynb`. Al final compara los transformers con las MobileNets de
`outputs/mobilenet/` (misma partición, modo de frames y receta).

Elegir siempre por **validación** (`val_mean_pmae`); test se informa pero no decide. `test_clean_mean_pmae` es el test
oficial sin los platos de `data/nutrition5k/excluded_dishes.csv`.

In [ ]:
import json
import os
from pathlib import Path

if Path("/workspace").is_dir():  # RunPod
    # The clone made by 02_experiments.ipynb: outputs/ links to the volume and data/ holds the frames.
    os.chdir("/root/thesis-multimodal-health-ai")
elif Path.cwd().parent.name == "notebooks":
    os.chdir("../..")

import matplotlib.pyplot as plt
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

SPLIT = "full"         # "full" o "pilot"
MODE = "complete"      # modo de frames a comparar
RECIPE = "fbn+clean"   # BN congelada (no cambia nada en los modelos con LayerNorm), sin platos erróneos, validación por grupos
RUNS_ROOT = Path("outputs/transformer")
MOBILENET_ROOT = Path("outputs/mobilenet")  # solo para la comparación final
# RUNS_ROOT holds <split>/<backbone>/<run>/ and <split>/seeds/
TARGETS = ("total_mass", "total_calories", "total_fat", "total_carb", "total_protein")
METRICS = {"pmae": "PMAE (%)", "mae": "MAE", "rmse": "RMSE", "rsquared": "R²"}  # R² is the only one where higher is better
SERIES = ("#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948")
INK, MUTED = "#0b0b0b", "#52514e"

In [ ]:
def read_json(path):
    return json.loads(path.read_text()) if path.is_file() else None


def run_dir(name):
    """outputs/transformer/<split>/<backbone>/<name>, from the run name alone."""
    return next(RUNS_ROOT.glob(f"*/*/{name}"))


def recipe(config):
    """fbn: frozen BatchNorm statistics (+clean: without excluded_dishes.csv); bn: updated statistics (earlier runs)."""
    if config.get("batchnorm") == "frozen":
        return "fbn+clean" if config.get("excluded_dishes") else "fbn"
    return "bn+cosine" if config.get("schedule") == "cosine" else "bn"


def load_runs(root=RUNS_ROOT):
    rows = []
    for config_path in sorted((root / SPLIT).glob("*/*/config.json")):
        run = config_path.parent
        config = read_json(config_path)
        history = pd.read_csv(run / "history.csv") if (run / "history.csv").is_file() else None
        val, test = read_json(run / "evaluation_val.json"), read_json(run / "evaluation.json")
        row = {
            "run": run.name,
            "path": run,
            "backbone": config["backbone"],
            "recipe": recipe(config),
            "unfreeze": "all" if config["unfreeze"] is None else str(config["unfreeze"]),
            "heads": config["heads"],
            "hidden": "x".join(map(str, config["hidden"])) or "0",
            "image_size": config["image_size"],
            "backbone_lr": config["backbone_lr"],
            "seed": config["seed"],
            "split": config["dataset_mode"],
            "mode": config["frame_mode"],
            "params_M": config["parameters"] / 1e6,
            "trainable_M": config["trainable_parameters"] / 1e6,
            "epochs_done": 0 if history is None else len(history),
            "best_epoch": None if history is None else int(history.loc[history["val_mean_pmae"].idxmin(), "epoch"]),
            "sec_per_epoch": None if history is None else history["seconds"].mean(),
            "gpu": config["environment"]["gpu"],
            "gpu_memory_gb": None if history is None else history["gpu_memory_gb"].max(),
            "status": "done" if test else "running",
            "val_mean_pmae": val["mean_pmae"] if val else None,
            "test_mean_pmae": test["mean_pmae"] if test else None,
            "test_clean_mean_pmae": clean_mean_pmae(run),
        }
        rows.append(row | metric_columns(val, test))
    return pd.DataFrame(rows)


def clean_mean_pmae(folder):
    """Test mean PMAE without the excluded dishes (evaluation_clean.json), if the run wrote it."""
    clean = read_json(folder / "evaluation_clean.json")
    return clean["mean_pmae"] if clean else None


def metric_columns(val, test):
    """Every metric per target, as val_<target>_<metric> and test_<target>_<metric>."""
    return {
        f"{prefix}_{target}_{metric}": evaluation["metrics"][target][metric] if evaluation else None
        for prefix, evaluation in (("val", val), ("test", test))
        for target in TARGETS
        for metric in METRICS
    }


def baseline_row():
    """The average baseline as one more row (no history), to compare with the models."""
    root = Path(f"outputs/average-baseline/{SPLIT}")
    val, test = read_json(root / "evaluation_val.json"), read_json(root / "evaluation.json")
    if not val:
        print("no evaluation_val.json for the baseline: run `python -m nutrition5k baseline` again")
    row = {
        "run": "average baseline", "backbone": "baseline", "status": "done", "params_M": 0, "trainable_M": 0,
        "val_mean_pmae": val["mean_pmae"] if val else None,
        "test_mean_pmae": test["mean_pmae"] if test else None,
        "test_clean_mean_pmae": clean_mean_pmae(root),
    }
    return pd.DataFrame([row | metric_columns(val, test)])


all_runs = load_runs()
assert len(all_runs), f"no runs in {RUNS_ROOT / SPLIT}; train them with 02_experiments.ipynb"
all_runs = all_runs[(all_runs["split"] == SPLIT) & (all_runs["mode"] == MODE)] if len(all_runs) else all_runs
runs = all_runs[all_runs["recipe"] == RECIPE]
BASELINE = baseline_row()
BASELINE_PMAE = BASELINE["val_mean_pmae"].iloc[0]
print(f"{len(all_runs)} runs ({len(runs)} {RECIPE}); baseline mean PMAE: "
      f"val {BASELINE_PMAE}, test {BASELINE['test_mean_pmae'].iloc[0]}")
runs.sort_values("val_mean_pmae")

In [ ]:
def select(recipe=RECIPE, **fixed):
    """Runs matching every given field, e.g. select(backbone="v4s", heads="single"); recipe=None keeps every recipe."""
    chosen = all_runs if recipe is None else all_runs[all_runs["recipe"] == recipe]
    for field, value in fixed.items():
        chosen = chosen[chosen[field].astype(str) == str(value)]
    return chosen.sort_values("val_mean_pmae")


def bar_pmae(chosen, label, title):
    """Validation mean PMAE per run (lower is better), with the validation baseline as reference."""
    done = chosen.dropna(subset=["val_mean_pmae"]).sort_values("val_mean_pmae", ascending=False)
    fig, ax = plt.subplots(figsize=(7, 0.45 * len(done) + 1.2))
    colors = [MUTED if name == "average baseline" else SERIES[0] for name in done["run"]]
    bars = ax.barh(done[label].astype(str), done["val_mean_pmae"], height=0.6, color=colors)
    ax.bar_label(bars, fmt="%.1f%%", padding=4, color=INK, fontsize=9)
    if BASELINE_PMAE:
        ax.axvline(BASELINE_PMAE, color=MUTED, linestyle="--", linewidth=1)
        ax.text(BASELINE_PMAE, len(done) - 0.4, " baseline (val)", color=MUTED, fontsize=8, va="bottom")
    ax.set_xlabel("Validation mean PMAE (%)", color=MUTED)
    ax.set_title(title, loc="left", color=INK)
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="x", color="#e6e6e3", linewidth=0.8)
    ax.set_axisbelow(True)
    plt.tight_layout()
    plt.show()


def curves(chosen, label, title):
    """Validation mean PMAE per epoch; at most eight runs per chart."""
    fig, ax = plt.subplots(figsize=(7, 4))
    # Colour follows the label value (sorted), not the ranking, so it is stable across views.
    colors = dict(zip(sorted(chosen[label].astype(str).unique()), SERIES))
    for _, row in chosen.sort_values(label, key=lambda values: values.astype(str)).iterrows():
        if str(row[label]) not in colors or pd.isna(row["path"]):  # the baseline has no history
            continue
        history = pd.read_csv(row["path"] / "history.csv")
        ax.plot(history["epoch"], history["val_mean_pmae"], color=colors[str(row[label])],
                linewidth=2, marker="o", markersize=5, label=str(row[label]))
        best = history.loc[history["val_mean_pmae"].idxmin()]
        ax.plot(best["epoch"], best["val_mean_pmae"], marker="o", markersize=11, fillstyle="none",
                color=colors[str(row[label])], markeredgewidth=2)
    ax.xaxis.get_major_locator().set_params(integer=True)
    ax.set_xlabel("Epoch", color=MUTED)
    ax.set_ylabel("Validation mean PMAE (%)", color=MUTED)
    ax.set_title(title, loc="left", color=INK)
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(color="#e6e6e3", linewidth=0.8)
    ax.legend(frameon=False, title=label)
    plt.tight_layout()
    plt.show()


def target_table(chosen, label, split="val"):
    """Every metric per target (columns) for each run (rows); the best value of each column is highlighted."""
    chosen = chosen.set_index(chosen[label].astype(str))
    columns = {}
    for metric, name in METRICS.items():
        for target in TARGETS:
            columns[(name, target.removeprefix("total_"))] = chosen[f"{split}_{target}_{metric}"]
        if metric == "pmae":
            columns[(name, "mean")] = chosen[f"{split}_mean_pmae"]
    table = pd.DataFrame(columns)
    higher = [column for column in table.columns if column[0] == METRICS["rsquared"]]
    lower = [column for column in table.columns if column not in higher]
    best = "font-weight: bold; background-color: #d7f0e3"
    return (table.style.format("{:.2f}", na_rep="—")
            .highlight_min(subset=lower, props=best)
            .highlight_max(subset=higher, props=best))


def bar_targets(chosen, label, title, split="val"):
    """PMAE per target (and the mean) for each run, side by side."""
    done = chosen.dropna(subset=[f"{split}_mean_pmae"]).sort_values(label, key=lambda values: values.astype(str))
    names = [target.removeprefix("total_") for target in TARGETS] + ["mean"]
    width = 0.8 / max(len(done), 1)
    fig, ax = plt.subplots(figsize=(8, 4))
    for i, (color, (_, row)) in enumerate(zip(SERIES, done.iterrows())):
        values = [row[f"{split}_{target}_pmae"] for target in TARGETS] + [row[f"{split}_mean_pmae"]]
        positions = [x + (i - (len(done) - 1) / 2) * width for x in range(len(names))]
        ax.bar(positions, values, width=width * 0.9, color=color, label=str(row[label]))
    ax.set_xticks(range(len(names)), names)
    ax.set_ylabel(f"{'Validation' if split == 'val' else 'Test'} PMAE (%)", color=MUTED)
    ax.set_title(title, loc="left", color=INK)
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", color="#e6e6e3", linewidth=0.8)
    ax.set_axisbelow(True)
    ax.legend(frameon=False, title=label)
    plt.tight_layout()
    plt.show()


def compare(chosen, label, title):
    """Everything needed to pick a winner at one stage: summary, per-target validation metrics and charts."""
    display(chosen[SUMMARY + [label]])
    if chosen["val_mean_pmae"].notna().any():
        display(target_table(chosen, label))
        bar_pmae(chosen, label, title)
        bar_targets(chosen, label, f"{title}: validation PMAE per target")
        curves(chosen, label, f"{title}: validation curves")


SUMMARY = ["run", "status", "epochs_done", "best_epoch", "val_mean_pmae", "test_mean_pmae", "test_clean_mean_pmae", "params_M", "trainable_M", "sec_per_epoch"]

## Etapa 1: backbones (fine-tuning completo, cabeza `single`, 512, 224 px, lr del backbone 1e-4, semilla 0)

Para cada backbone: resumen (`best_epoch` es la época con el menor `val_mean_pmae`, la que se guarda en `best.pt`
y se evalúa), tabla de validación con PMAE, MAE, RMSE y R² por objetivo (en verde el mejor de cada columna), PMAE por
objetivo en barras y curvas de validación (el círculo grande marca la mejor época). La fila `baseline` es la línea base
de la media sobre la misma validación.

In [ ]:
STAGE1 = dict(heads="single", hidden="512", image_size=224, backbone_lr=1e-4, seed=0)
# Full fine-tuning for every backbone, plus the frozen DINOv2 (its features are meant to work frozen).
stage1 = pd.concat([select(unfreeze="all", **STAGE1), select(backbone="dinov2_s", unfreeze="0", **STAGE1)].sort_values("val_mean_pmae")
stage1 = stage1.assign(model=stage1["backbone"] + " ft " + stage1["unfreeze"])
compare(pd.concat([stage1, BASELINE.assign(model="baseline")]), "model", f"Stage 1 ({RECIPE})")

### Etapa 1: coste computacional

Lo que cuesta cada backbone, para elegir también por tamaño y cómputo (el objetivo es un dispositivo edge):

- `params_M`: parámetros en millones (modelo completo: backbone, tronco y cabezas).
- `size_MB`: tamaño de `best.pt` (pesos en float32, 4 bytes por parámetro); con int8 sería ~4 veces menor.
- `MMACs`: millones de multiplicaciones-acumulaciones por imagen (convoluciones y capas lineales), a la resolución
  del run. Es el coste teórico de una inferencia y no depende del hardware.
- Los transformers corren siempre a 224 px.
- `cpu_ms`: latencia mediana de una imagen (batch 1) en la CPU de esta máquina, con pesos aleatorios. Depende del
  hardware y de cómo se ejecuta cada operación, así que solo sirve para comparar backbones entre sí en la misma máquina.
- `gpu_memory_gb` y `sec_per_epoch`: pico de memoria de GPU y segundos por época **del entrenamiento** (batch 32). El
  entrenamiento está limitado por la CPU que decodifica los JPEG, así que `sec_per_epoch` casi no distingue backbones.

In [ ]:
import time

import torch
from torch.utils.flop_counter import FlopCounterMode

from nutrition5k.models.nutrition_net import NutritionNet


def inference_cost(row, repeats=30):
    """MACs and median CPU latency (batch 1) of a run's architecture; the cost does not depend on the weights."""
    hidden = () if row["hidden"] == "0" else tuple(map(int, row["hidden"].split("x")))
    model = NutritionNet(row["backbone"], row["heads"], hidden, pretrained=False).eval()
    size = int(row["image_size"])
    image = torch.randn(1, 3, size, size)
    with torch.no_grad():
        with FlopCounterMode(display=False) as counter:
            model(image)
        for _ in range(5):  # warm-up
            model(image)
        times = []
        for _ in range(repeats):
            began = time.perf_counter()
            model(image)
            times.append(time.perf_counter() - began)
    # The counter reports FLOPs; one multiply-accumulate is two of them.
    return {"MMACs": counter.get_total_flops() / 2 / 1e6, "cpu_ms": 1000 * sorted(times)[repeats // 2]}


stage1 = stage1[stage1["status"] == "done"]
cost = stage1[["model", "val_mean_pmae", "params_M"]].assign(
    size_MB=[(path / "best.pt").stat().st_size / 1e6 for path in stage1["path"]]
)
cost = cost.join(pd.DataFrame([inference_cost(row) for _, row in stage1.iterrows()], index=stage1.index))
cost = cost.join(stage1[["gpu_memory_gb", "sec_per_epoch", "gpu"]]).set_index("model")
display(cost.round(2))

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(cost["MMACs"], cost["val_mean_pmae"], s=64, color=SERIES[0], zorder=3)
for backbone, row in cost.iterrows():
    ax.annotate(backbone, (row["MMACs"], row["val_mean_pmae"]), xytext=(7, 4), textcoords="offset points",
                color=INK, fontsize=9)
ax.set_xlabel("MACs per image (millions)", color=MUTED)
ax.set_ylabel("Validation mean PMAE (%)", color=MUTED)
ax.set_title("Accuracy vs compute (lower left is better)", loc="left", color=INK)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(color="#e6e6e3", linewidth=0.8)
ax.set_axisbelow(True)
plt.tight_layout()
plt.show()

## Etapa 2: ablaciones (rellenar con los mismos `BACKBONES` y `BEST_*` que en `02_experiments.ipynb`)

Cada backbone sigue su propio camino greedy. Paso 1: profundidad de fine-tuning (`unfreeze`: últimos bloques en DeiT y
DINOv2, últimas etapas en FastViT y Swin). Paso 2: lr del backbone. Diferencias de menos de ~1 punto son empate.

In [ ]:
BACKBONES = ()
BEST_UNFREEZE = {}  # como texto, p. ej. {"dinov2_s": "3"}

for backbone in BACKBONES:
    steps = {
        "Step 1: fine-tuning depth": ("unfreeze", select(backbone=backbone, heads="single", hidden="512", backbone_lr=1e-4, seed=0)),
        "Step 2: backbone learning rate": ("backbone_lr", select(backbone=backbone, unfreeze=BEST_UNFREEZE.get(backbone), heads="single", hidden="512", seed=0)),
    }
    for title, (label, chosen) in steps.items():
        print(f"{backbone} - {title}")
        if chosen[label].nunique() < 2:
            print("  nothing to compare yet")
            continue
        compare(chosen, label, f"{backbone} - {title}")

## Error por objetivo (test) de todas las ejecuciones terminadas

Ordenadas por validación. Solo se informa: no sirve para elegir.

In [ ]:
target_table(runs[runs["status"] == "done"].sort_values("val_mean_pmae"), "run", split="test")

## Evolución por objetivo de una ejecución

Comprueba si un objetivo empeora mientras la media mejora. La línea negra es la media; la vertical marca la época elegida.

In [ ]:
def target_curves(run):
    history = pd.read_csv(run_dir(run) / "history.csv")
    fig, ax = plt.subplots(figsize=(7, 4))
    for color, target in zip(SERIES, TARGETS):
        ax.plot(history["epoch"], history[f"val_{target}_pmae"], color=color, linewidth=2,
                marker="o", markersize=5, label=target.removeprefix("total_"))
    ax.plot(history["epoch"], history["val_mean_pmae"], color=INK, linewidth=2, label="mean")
    ax.axvline(history["val_mean_pmae"].idxmin(), color=MUTED, linestyle="--", linewidth=1)
    ax.xaxis.get_major_locator().set_params(integer=True)
    ax.set_xlabel("Epoch", color=MUTED)
    ax.set_ylabel("Validation PMAE (%)", color=MUTED)
    ax.set_title(run, loc="left", color=INK, fontsize=10)
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(color="#e6e6e3", linewidth=0.8)
    ax.legend(frameon=False)
    plt.tight_layout()
    plt.show()


RUN = runs.sort_values("val_mean_pmae")["run"].iloc[0]  # o escribir el nombre de una ejecución
target_curves(RUN)

## Etapa 3: media ± SD entre semillas

Una fila por configuración con más de una semilla.

In [ ]:
config_columns = ["backbone", "unfreeze", "heads", "hidden", "image_size", "backbone_lr"]
done = runs[runs["status"] == "done"]
seeds = done.groupby(config_columns).agg(
    seeds=("seed", "nunique"),
    val_mean=("val_mean_pmae", "mean"),
    val_sd=("val_mean_pmae", "std"),
    test_mean=("test_mean_pmae", "mean"),
    test_sd=("test_mean_pmae", "std"),
)
seeds[seeds["seeds"] > 1].sort_values("val_mean")

## Métricas de test de los modelos finales

MAE, PMAE, RMSE y R² por objetivo, con la línea base como referencia. Test solo se informa: los modelos se eligen por validación.
Por defecto muestra los tres mejores runs por validación; se puede escribir la lista a mano.

In [ ]:
FINAL_RUNS = list(runs[runs["status"] == "done"].sort_values("val_mean_pmae")["run"].head(3))


def test_metrics(run_names):
    """One row per model and target, from each run's evaluation.json and the baseline's."""
    sources = {name: run_dir(name) / "evaluation.json" for name in run_names}
    sources["average baseline"] = Path(f"outputs/average-baseline/{SPLIT}/evaluation.json")
    rows = []
    for model, path in sources.items():
        evaluation = read_json(path)
        if not evaluation:
            continue
        for target in TARGETS:
            metric = evaluation["metrics"][target]
            rows.append({
                "model": model, "target": target.removeprefix("total_"), "unit": metric["unit"],
                "MAE": metric["mae"], "PMAE (%)": metric["pmae"], "RMSE": metric["rmse"], "R²": metric["rsquared"],
            })
        rows.append({"model": model, "target": "mean", "PMAE (%)": evaluation["mean_pmae"]})
    return pd.DataFrame(rows).set_index(["model", "target"])


test_metrics(FINAL_RUNS).round(2)

### Media ± SD entre semillas (test)

Lee `outputs/transformer/<split>/seeds/*.json`, que escribe `train --all-seeds`.

In [ ]:
rows = []
for path in sorted((RUNS_ROOT / SPLIT / "seeds").glob("*.json")):
    if f"-{MODE}-{SPLIT}" not in path.stem:
        continue
    summary = read_json(path)
    row = {"configuration": path.stem}
    for split, prefix in (("validation", "val"), ("test", "test")):
        mean_pmae = summary[split]["mean_pmae"]
        row[f"{prefix}_mean_pmae"] = f"{mean_pmae['mean']:.2f} ± {mean_pmae['sd']:.2f}"
    for target in TARGETS:
        pmae = summary["test"]["metrics"][target]["pmae"]
        row[f"test_{target.removeprefix('total_')}_pmae"] = f"{pmae['mean']:.2f} ± {pmae['sd']:.2f}"
    rows.append(row)
pd.DataFrame(rows)

## Transformers frente a MobileNets

El mejor run de cada backbone por validación (semilla 0), en las dos familias, con su coste de inferencia
(`inference_cost`, definido en la sección de coste computacional). Es la pregunta del TFG: qué error se obtiene a
cambio de qué coste. Comparar las configuraciones finales con 3 semillas cuando estén (etapa 3 de cada familia).

In [ ]:
mobilenets = load_runs(MOBILENET_ROOT)
if len(mobilenets):
    mobilenets = mobilenets[(mobilenets["split"] == SPLIT) & (mobilenets["mode"] == MODE) & (mobilenets["recipe"] == RECIPE)]
both = pd.concat([runs.assign(family="transformer"), mobilenets.assign(family="mobilenet")])
both = both[(both["status"] == "done") & (both["seed"] == 0)]
best = both.loc[both.groupby("backbone")["val_mean_pmae"].idxmin()]
best = best.join(pd.DataFrame([inference_cost(row) for _, row in best.iterrows()], index=best.index))
display(best[["family", "backbone", "run", "val_mean_pmae", "test_mean_pmae", "params_M", "MMACs", "cpu_ms"]]
        .sort_values("val_mean_pmae").round(2))

fig, ax = plt.subplots(figsize=(7, 4))
for color, (family, group) in zip(SERIES, best.groupby("family")):
    ax.scatter(group["MMACs"], group["val_mean_pmae"], s=64, color=color, zorder=3, label=family)
    for _, row in group.iterrows():
        ax.annotate(row["backbone"], (row["MMACs"], row["val_mean_pmae"]), xytext=(7, 4),
                    textcoords="offset points", color=INK, fontsize=9)
ax.set_xscale("log")
ax.set_xlabel("MACs per image (millions, log scale)", color=MUTED)
ax.set_ylabel("Validation mean PMAE (%)", color=MUTED)
ax.set_title("Best run per backbone: accuracy vs compute (lower left is better)", loc="left", color=INK)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(color="#e6e6e3", linewidth=0.8)
ax.set_axisbelow(True)
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

## Galería de platos de test

Para un run: los platos con menor error, los de mayor error y algunos al azar, con su primer frame, el valor real y el predicho.
El error de un plato es la media, sobre los cinco objetivos, de |error| / media real del objetivo en test (como el PMAE).
Necesita los frames en `data/` (en local o en el pod).

In [ ]:
def gallery(run, count=4, seed=0):
    try:
        from nutrition5k import load_dataset

        dataset = load_dataset(Path("data/nutrition5k"), full=SPLIT == "full", frame_mode=MODE)
    except Exception as exc:  # no frames in data/
        print(f"the gallery needs the frames in data/: {exc}")
        return
    records = {record.dish_id: record for record in dataset.test if record.frame_paths}
    predicted = pd.read_csv(run_dir(run) / "predictions.csv").set_index("dish_id")
    predicted = predicted.loc[predicted.index.intersection(list(records))]
    truth = pd.DataFrame({dish: records[dish].targets for dish in predicted.index}, index=TARGETS).T
    error = ((predicted[list(TARGETS)] - truth).abs() / truth.mean()).mean(axis=1).sort_values()
    rest = error.iloc[count:-count]
    groups = {
        "lowest error": error.index[:count],
        "highest error": error.index[-count:][::-1],
        "random": rest.sample(min(count, len(rest)), random_state=seed).index,
    }
    fig, axes = plt.subplots(len(groups), count, figsize=(3.6 * count, 3.4 * len(groups)), squeeze=False)
    for row, (title, dishes) in zip(axes, groups.items()):
        for ax, dish in zip(row, dishes):
            ax.imshow(plt.imread(records[dish].frame_paths[0]))
            lines = [
                f"{target.removeprefix('total_')}: {truth.loc[dish, target]:.0f} → {predicted.loc[dish, target]:.0f}"
                for target in TARGETS
            ]
            ax.set_title(f"{title}\n{dish}", loc="left", color=INK, fontsize=9)
            ax.set_xlabel("\n".join(lines) + f"\nerror {error[dish]:.0%}", color=MUTED, fontsize=8, loc="left")
            ax.set_xticks([]), ax.set_yticks([])
        for ax in row[len(dishes):]:
            ax.axis("off")
    fig.suptitle(f"{run}: real → predicted (g, kcal)", x=0.01, ha="left", color=INK)
    plt.tight_layout()
    plt.show()


gallery(RUN)  # RUN se define arriba (el mejor por validación); se puede escribir otro nombre